# 手写一个简单的gpt，理解其原理

## 数据集准备

In [ ]:
# 数据集准备
# 网址：https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt



123


## 构建分词器tokenize
把词转换为一个整数序列

In [ ]:
#简单实现endocer和decoder




#谷歌的sentence piece

#字节的tiktoken

In [ ]:
#划分训练集和验证集

block_size指的是用于预测的最大上下文长度，是训练集每次训练时从数据集中提取block_size的token输入给transformer网络(一个批次)

设计context和target，context是每次训练时输入给trnsformer的内容，每一次训练context都把之前所有的token一起输入给transformer。target是每次训练时需要预测的内容，即同一时刻context的下一个token，并且每轮训练都进行block_size次(一个批次)

In [ ]:
# 

引入批处理维度

构造context和target矩阵

## 构造一个简单的二元语言模型，获得预测结果、logits以及loss等

用一个generate函数用于拓宽时间维度

开始训练这个模型


将这个简单的二元语言模型整理和优化后整理为bigram.py脚本，可以在此基础上加进行优化

## transformer架构的实现




### 自注意力模块的实现

可以高效实现self-attention的数学技巧

实现token之间的通信，要保证当前时刻的token无法与之后的token联系，只能与之前的token通信，可以采取对小于等于当前时刻的token进行加权和、取均值等方式得到一个特征向量，它包含了历史token信息，但要尽可能的不浪费这些信息，同时也要注意token之间的空间顺序关系。

In [ ]:
# 当前时刻，对历史token取均值作为xbow向量的一个元素

#用矩阵乘法高效计算

# 

自注意力模块的优化实现在v2.py

位置编码

自注意力机制的核心：从历史数据中获得所有token之间的相似度
实现方法：每个token向量X都会生成三个向量，分别为query、kay向量、value向量，query可以大致理解为“我在寻找什么”，key向量可以大致理解为“我包含什么信息”，value向量代表着“需要在不同节点之间进行聚合的信息”，v应该就包含了token本身的信息。然后就通过Q和K向量的点积来获得两个token之间的相似度(一个token的Q向量会其他所有token的k向量进行点积)，接着得到的相似度(向量/矩阵)会经过mask和softmax处理后和v相乘，就得到了自注意力模块的输出

关于注意力机制的补充：
1.注意力机制本质上是一种节点间(有向图)的信息传递机制，每个节点都携带一个信息向量(v?，通过加权求和的方式聚合所有指向该节点的其他节点所传递的信息。图结构是：当前节点由自身以及之前所有节点共
2.注意力本质上是在图中对一组向量进行操作，节点不知道自己的空间位置，因此才需要位置编码额外赋予每个节点位置信息
3.在decode模块中，必须mask保证当前时刻节点无法得到后面节点的信息，但在incoder模块中，对于某些特定任务，可以让过去和未来的节点互相通信，可以不要mask

In [ ]:
# 自注意力头

# 生成Q,K,V，获得相似度

# mask和softmax

# out

除了自注意力之外，还有交叉注意力，相比于自注意力的QKV来源于同一个输入X，交叉注意力的Q和kv来自不同数据，Q来自上一个多头注意力模块的输出，k和v来自编码器的输出。交叉注意力机制的作用是：解码过程不仅来自于当前解码的历史信息(Q),同时还参考了已完整编码的来自编码器的提示信息

注意力缩放:就是计算注意力的公式中在qk加权和之后除以根号下的维度，目的是控制方差。这是为了在接下来输入给softmax时，避免因为某个值过大，导致softmax后这个值趋近于1而其他值趋近于0


实现多头注意力

加上残差

Add norm

后记：这里实现的只是一个解码器transformer，没有实现编码器以及交叉注意力的部分。因为数据集给的是英文，而当前目的只是模仿其内容生成同为英文的文字，因此使用纯解码器架构

## nano gpt简介

nanogpt的模型定义部分(module.py)和这些代码高度相似，但在一些地方有些差别

多头注意力的实现上在QKV增加了一个多头注意力的维度、用了Gelu而不是relu、多了一些额外的步骤

最初训练的是预训练的模型，预训练之后的模型不能像大模型一样回答问题，它只会模仿数据集来生成补全文字。第二阶段是微调，这才能让模型真正成为一个助手，gpt-3的做法是收集专门类似于助手行为模式的训练数据。之后再经过更多的训练策略(包括评估，强化学习训练等)，最终让模型从一个文档补全工具，转变成一个问答系统

nanogpt则关注预训练阶段